In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from lib.matrix_dataset import MatrixDataset
from recommenders.lmf import LogisticMatrixFactorization

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    device = mps_device
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

if device == torch.device("cuda"):
    dtype = torch.float32
    print("Using CUDA.")
elif device == torch.device("cpu"):
    dtype = torch.float64
    print("Using CPU.")
elif device == torch.device("mps"):
    dtype = torch.float32
    print("Using MPS.")

# device = "cpu"
# dtype = torch.float64

tensor([1.], device='mps:0')
Using MPS.


# Import data

In [3]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [4]:
df_matrix_mf = df.copy()
df_matrix_mf.loc[df_matrix_mf["type"] == "liked_track", "affinity"] = 0.5
df_matrix_mf.loc[df_matrix_mf["type"] == "playlist", "affinity"] = 0.3

used_types = ["top_track", "liked_track", "playlist"]
used_types = ["top_track"]
df_matrix_mf = df_matrix_mf[df["type"].isin(used_types)]
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]


,username,id,affinity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
0,jaslkh,75rqqKvzJCGv2oq9C4yFDt,1.00,0.630,0.374,0.2270,0.8610,0.000075,0.2940,0.3730,104.955,-9.007,196426,2023,63
1,jaslkh,2FYGZDfsAnNsrm1gVbyKnG,0.98,0.827,0.768,0.2650,0.7900,0.000024,0.4970,0.7340,99.988,-5.702,137533,2022,70
2,jaslkh,4kroNlz8BTfswE4M0i3YCh,0.96,0.583,0.297,0.4060,0.8790,0.000000,0.1270,0.2910,124.279,-11.273,162906,2022,61
3,jaslkh,2N3YZ075lq9z1ObaAiX6l1,0.94,0.651,0.327,0.4020,0.9280,0.000000,0.2250,0.5510,135.325,-10.070,89749,2022,54
4,jaslkh,2SiAcexM2p1yX6joESbehd,0.92,0.555,0.634,0.2730,0.1290,0.000002,0.1880,0.5550,170.228,-5.522,174044,2023,73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12318,dany,4GRMPDD3V6rM2BlmRdYCUJ,0.10,0.491,0.699,0.0924,0.1620,0.864000,0.1050,0.0762,130.325,-10.500,214000,2022,41
12319,dany,17Xof0GRZZfS7ZgjUJ27pH,0.08,0.271,0.503,0.0314,0.7500,0.000102,0.0941,0.0742,127.960,-4.142,241821,2020,50
12320,dany,2Rd4eJ4KwXQQn2sMSToyUM,0.06,0.319,0.236,0.0318,0.8980,0.000016,0.2720,0.2540,162.351,-9.644,156091,2021,6
12321,dany,4fPBB44eDH71YohayI4eKV,0.04,0.630,0.908,0.0326,0.0238,0.592000,0.1160,0.9680,120.522,-2.420,189226,2006,75


In [5]:
df_matrix_mf = pd.read_csv(os.path.join(data_path.DATA_PATH, "artificial_data.csv"), index_col=0)

In [6]:
df_matrix_mf["affinity"] *= 1

In [7]:
matrix_mf = MatrixDataset(df_matrix_mf, "username", "id", "affinity", device=device, dtype=dtype)
R = matrix_mf.R
R.to(device)
R.shape

torch.Size([497, 53477])

In [8]:
alpha = matrix_mf.compute_alpha().item()
R *= 0.1
alpha

353.5196533203125

In [9]:
num_latent_factors = 3
lmf = LogisticMatrixFactorization(
    counts=R,
    num_factors=num_latent_factors,
    reg_param=0.01,
    device=device,
    dtype=dtype,
)
lmf.to(device)

num_epochs = 100
lmf.train_model(
    num_epochs=num_epochs,
    learning_rate=1,
    log_interval=1,
    tqdm=False
)

Epoch: 1, log-likelihood: 6936001.5000, MPR: 0.4993
Epoch: 2, log-likelihood: 2934923.5000, MPR: 0.4989
Epoch: 3, log-likelihood: 621249.5000, MPR: 0.4917
Epoch: 4, log-likelihood: 299230.8750, MPR: 0.4807
Epoch: 5, log-likelihood: 203624.9219, MPR: 0.4715
Epoch: 6, log-likelihood: 159124.4688, MPR: 0.4635
Epoch: 7, log-likelihood: 134170.0781, MPR: 0.4571
Epoch: 8, log-likelihood: 118545.8984, MPR: 0.4511
Epoch: 9, log-likelihood: 108026.9062, MPR: 0.4463
Epoch: 10, log-likelihood: 100553.5469, MPR: 0.4419
Epoch: 11, log-likelihood: 95038.3906, MPR: 0.4381
Epoch: 12, log-likelihood: 90832.1406, MPR: 0.4347
Epoch: 13, log-likelihood: 87550.2578, MPR: 0.4317
Epoch: 14, log-likelihood: 84927.8672, MPR: 0.4289
Epoch: 15, log-likelihood: 82802.5859, MPR: 0.4264
Epoch: 16, log-likelihood: 81051.3438, MPR: 0.4241
Epoch: 17, log-likelihood: 79592.5469, MPR: 0.4220
Epoch: 18, log-likelihood: 78361.1641, MPR: 0.4200
Epoch: 19, log-likelihood: 77312.8594, MPR: 0.4181
Epoch: 20, log-likelihood: 7

In [10]:
lmf.save("models", "lmf")

In [11]:
px.line(x=range(len(lmf.losses)), y=lmf.losses.cpu(), title="Loss").show()
px.line(x=range(len(lmf.mprs)), y=lmf.mprs.cpu(), title="MPRS").show()

In [13]:
user_id = matrix_mf.usernames_to_ids(["paul"])[0]
print(user_id)

# Get the top 10 recommendations for the user
top_10_ids, top_10_ids_scores = lmf.recommend(user_id, top_k=20, filter_user_items=True)

# Get the top 10 recommendations for the user
matrix_mf.item_ids_to_df(top_10_ids)

-1


,id,username,danceability,duration_ms,instrumentalness,affinity
15,Couple,Stephanie Ross,1.118199,0.971794,1.014813,1.092027
94,Couple,Stephanie Ross,1.025700,0.899719,0.909549,0.868304
337,Property,Amanda Bautista,1.032118,0.999721,1.133375,0.915211
343,Opportunity,Amanda Bautista,1.093076,0.801353,0.914944,0.829884
437,Individual,Amanda Bautista,1.003964,0.907915,1.029226,0.996542
...,...,...,...,...,...,...
74676,Soldier,Dr. William Schaefer,1.122970,0.924892,0.879718,1.029970
74709,Scientist,Tyler Patton,1.053394,1.124458,0.795413,0.857619
74794,Determine,Tyler Patton,0.986698,1.067318,1.064204,1.029139
74822,Character,Tyler Patton,0.976955,0.918152,1.091632,1.062274


In [15]:
if lmf.num_factors <= 3:
    df_tracks = df_matrix_mf.copy()
    df_tracks = df_tracks.sample(frac=1) # Shuffle the dataframe
    df_tracks = df_tracks[:150] # Keep only 1000 tracks

    # Add item latent factors to the dataframe
    items_latent_columns = [f"track_latent_{i}" for i in range(lmf.num_factors)]
    for track_id in df_tracks["id"].unique():
        latent_factors = lmf.get_item_factors([matrix_mf.itemnames_to_ids([track_id])[0]]).tolist()
        df_tracks.loc[df_tracks["id"] == track_id, items_latent_columns] = latent_factors

    # Add user latent factors to the dataframe
    users_latent_columns = [f"latent_factor_{i}" for i in range(lmf.num_factors)]
    for user_id in df_tracks["username"].unique():
        latent_factors = lmf.get_user_factors([matrix_mf.usernames_to_ids([user_id])[0]]).tolist()
        df_tracks.loc[df_tracks["username"] == user_id, users_latent_columns] = latent_factors

    plotting.plot_latent_space(
        df=df_tracks,
        color=df_tracks["username"],
        text=df_tracks["username"],
        latent_columns=items_latent_columns,
    ).show()

    plotting.plot_latent_space(
        df=df_tracks,
        color=df_tracks["username"],
        text=df_tracks["username"],
        latent_columns=users_latent_columns,
    ).show()